# BirdCLEF 2026 — Submission Notebook v5
**Architecture**: Perch v2 ONNX → PCA(256) → MLP probe (256→128→234) + genus proxy + Perch blend

**Improvements over v4.2**: MLP probe (vs LogReg), soundscape augmentation, tuned blend=0.10, no hard prior suppression.

In [1]:
import subprocess
subprocess.run([
    'pip', 'install',
    '/kaggle/input/datasets/eduarddehelean/perch-v2-onnx/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl',
    '--quiet'
], check=True)
print('onnxruntime installed')

onnxruntime installed


## 1. Imports

In [2]:
import os, warnings, gc
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from scipy.special import expit

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import soundfile as sf
import torch
import torch.nn as nn
import torchaudio
import onnxruntime as ort
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
import joblib

if torch.cuda.is_available():
    TORCH_DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    TORCH_DEVICE = torch.device('mps')
else:
    TORCH_DEVICE = torch.device('cpu')

available_providers = ort.get_available_providers()
if 'CUDAExecutionProvider' in available_providers:
    ONNX_PROVIDERS = ['CUDAExecutionProvider', 'CPUExecutionProvider']
else:
    ONNX_PROVIDERS = ['CPUExecutionProvider']

print('torch device:', TORCH_DEVICE)
print('ONNX providers:', ONNX_PROVIDERS)

torch device: cpu
ONNX providers: ['CPUExecutionProvider']


## 2. Config

In [3]:
BASE_DIR   = Path('/kaggle/input/competitions/birdclef-2026')
MODELS_DIR = Path('/kaggle/input/datasets/eduarddehelean/perch-v2-onnx')
EMB_DIR    = Path('/kaggle/input/datasets/eduarddehelean/birdclef2026-embeddings')
OUTPUT_DIR = Path('/kaggle/working')
PROBE_DIR  = OUTPUT_DIR / 'probes'
SOUNDSCAPE_EMB_DIR = OUTPUT_DIR / 'soundscape_emb_cache'
for d in [PROBE_DIR, SOUNDSCAPE_EMB_DIR]:
    d.mkdir(exist_ok=True)

TRAIN_CSV             = BASE_DIR / 'train.csv'
TAXONOMY_CSV          = BASE_DIR / 'taxonomy.csv'
SOUNDSCAPE_LABELS_CSV = BASE_DIR / 'train_soundscapes_labels.csv'
SAMPLE_SUB_CSV        = BASE_DIR / 'sample_submission.csv'
TRAIN_SOUNDSCAPES_DIR = BASE_DIR / 'train_soundscapes'
TEST_SOUNDSCAPES_DIR  = BASE_DIR / 'test_soundscapes'
PERCH_ONNX_PATH       = MODELS_DIR / 'perch_v2.onnx'
PERCH_LABELS_PATH     = MODELS_DIR / 'labels.csv'
EMB_CACHE             = EMB_DIR / 'embeddings.npz'

SAMPLE_RATE  = 32_000
CLIP_SAMPLES = SAMPLE_RATE * 5
N_WINDOWS    = 12
SEED         = 42
PERCH_BLEND  = 0.10
SUPPRESS_FACTOR = 1.0   # 1.0 = no prior suppression; 0.0 = hard zero

print('Competition data :', BASE_DIR.exists())
print('Perch ONNX       :', PERCH_ONNX_PATH.exists())
print('Embeddings cache :', EMB_CACHE.exists())

Competition data : True
Perch ONNX       : True
Embeddings cache : True


## 3. Metadata

In [4]:
train_df          = pd.read_csv(TRAIN_CSV)
taxonomy          = pd.read_csv(TAXONOMY_CSV)
soundscape_labels = pd.read_csv(SOUNDSCAPE_LABELS_CSV).drop_duplicates(['filename','start','end'])
sample_sub        = pd.read_csv(SAMPLE_SUB_CSV)

TARGET_CLASSES = [c for c in sample_sub.columns if c != 'row_id']
NUM_CLASSES    = len(TARGET_CLASSES)
label2idx      = {lbl: i for i, lbl in enumerate(TARGET_CLASSES)}

print(f'Target classes: {NUM_CLASSES}')
print(f'Labeled soundscapes: {soundscape_labels["filename"].nunique()}')

Target classes: 234
Labeled soundscapes: 66


## 4. Audio & Perch Utilities

In [5]:
def load_audio(path, target_sr=SAMPLE_RATE):
    audio, sr = sf.read(path, dtype='float32', always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != target_sr:
        audio_t = torch.from_numpy(audio).unsqueeze(0)
        audio_t = torchaudio.functional.resample(audio_t, sr, target_sr)
        audio   = audio_t.squeeze(0).numpy()
    return audio

def split_into_clips(audio, clip_samples=CLIP_SAMPLES, max_clips=N_WINDOWS):
    clips = []
    for start in range(0, len(audio), clip_samples):
        chunk = audio[start:start + clip_samples]
        if len(chunk) < clip_samples:
            chunk = np.pad(chunk, (0, clip_samples - len(chunk)))
        clips.append(chunk)
        if len(clips) >= max_clips:
            break
    return clips

perch_labels = pd.read_csv(PERCH_LABELS_PATH, header=None, names=['scientific_name'])

sci2perch = {}
for _, row in taxonomy.iterrows():
    match = perch_labels[perch_labels['scientific_name'] == row['scientific_name']]
    if not match.empty:
        sci2perch[str(row['primary_label'])] = match.index[0]

MATCHED_LABELS    = sorted(sci2perch.keys(), key=lambda x: label2idx[x])
MATCHED_PERCH_IDX = [sci2perch[lbl] for lbl in MATCHED_LABELS]

GENUS_PROXY_LABELS, GENUS_PROXY_IDX_SETS = [], []
for _, row in taxonomy.iterrows():
    lbl = str(row['primary_label'])
    if lbl in sci2perch:
        continue
    genus   = str(row['scientific_name']).split()[0]
    matches = perch_labels[perch_labels['scientific_name'].str.startswith(genus + ' ')].index.tolist()
    if matches:
        GENUS_PROXY_LABELS.append(lbl)
        GENUS_PROXY_IDX_SETS.append(matches)

print(f'Matched in Perch  : {len(sci2perch)}/234')
print(f'Genus proxy covers: {len(GENUS_PROXY_LABELS)} unmatched species')

sess_options = ort.SessionOptions()
sess_options.inter_op_num_threads = 4
sess_options.intra_op_num_threads = 4
perch_session = ort.InferenceSession(str(PERCH_ONNX_PATH), sess_options=sess_options, providers=ONNX_PROVIDERS)
input_name   = perch_session.get_inputs()[0].name
output_names = [o.name for o in perch_session.get_outputs()]
_dummy       = np.zeros((1, CLIP_SAMPLES), dtype=np.float32)
_outs        = dict(zip(output_names, perch_session.run(output_names, {input_name: _dummy})))
_logit_name  = max(output_names, key=lambda n: _outs[n].shape[-1])
print(f'Perch ready — "{_logit_name}" {_outs[_logit_name].shape}')
print(f'ONNX active provider: {perch_session.get_providers()}')

_BATCHED_PERCH = False
try:
    _bo = perch_session.run(output_names, {input_name: np.zeros((N_WINDOWS, CLIP_SAMPLES), dtype=np.float32)})
    if dict(zip(output_names, _bo))[_logit_name].shape[0] == N_WINDOWS:
        _BATCHED_PERCH = True
except Exception:
    pass
print(f'Batched Perch: {_BATCHED_PERCH}')

def get_perch_embeddings(clips):
    inp = np.stack(clips).astype(np.float32)
    if _BATCHED_PERCH:
        outs        = dict(zip(output_names, perch_session.run(output_names, {input_name: inp})))
        full_logits = outs[_logit_name]
        emb         = outs['embedding']
    else:
        emb_list, logit_list = [], []
        for clip in inp:
            o = dict(zip(output_names, perch_session.run(output_names, {input_name: clip[np.newaxis]})))
            emb_list.append(o['embedding'])
            logit_list.append(o[_logit_name])
        emb, full_logits = np.vstack(emb_list), np.vstack(logit_list)
    logits = full_logits[:, MATCHED_PERCH_IDX] if full_logits.shape[-1] > max(MATCHED_PERCH_IDX) else full_logits
    proxy_row    = [full_logits[:, idx_set].mean(axis=1, keepdims=True) for idx_set in GENUS_PROXY_IDX_SETS]
    proxy_logits = np.concatenate(proxy_row, axis=1) if proxy_row else np.zeros((len(clips), 0), dtype=np.float32)
    return emb, logits, proxy_logits

def load_soundscape_embeddings(path):
    cache_file = SOUNDSCAPE_EMB_DIR / f'{path.stem}_w{N_WINDOWS}.npz'
    if cache_file.exists():
        c = np.load(cache_file)
        return c['emb'], c['logits'], c['proxy_logits'], True
    clips = split_into_clips(load_audio(path), max_clips=N_WINDOWS)
    emb, logits, proxy_logits = get_perch_embeddings(clips)
    np.savez_compressed(cache_file, emb=emb, logits=logits, proxy_logits=proxy_logits)
    return emb, logits, proxy_logits, False

Matched in Perch  : 203/234
Genus proxy covers: 6 unmatched species
Perch ready — "label" (1, 14795)
ONNX active provider: ['CPUExecutionProvider']
Batched Perch: True


## 5. Site / Hour Priors

In [6]:
def parse_site_hour(filename):
    parts = filename.replace('.ogg', '').split('_')
    site  = next((p for p in parts if p.startswith('S') and p[1:].isdigit()), 'unknown')
    try:    hour = int(parts[-1][:2])
    except: hour = -1
    return site, hour

def build_site_hour_priors(df):
    priors = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
    for _, row in df.iterrows():
        site, hour = parse_site_hour(row['filename'])
        for lbl in str(row['primary_label']).split(';'):
            lbl = lbl.strip()
            if lbl in label2idx:
                priors[site][hour][lbl] += 1
    return priors

def apply_prior_suppression(scores, site, hour, priors, suppress_factor=SUPPRESS_FACTOR):
    if suppress_factor >= 1.0:
        return scores
    scores  = scores.copy()
    active  = set(priors.get(site, {}).get(hour, {}).keys())
    for i, lbl in enumerate(TARGET_CLASSES):
        if lbl not in active:
            scores[:, i] *= suppress_factor
    return scores

site_hour_priors = build_site_hour_priors(soundscape_labels)
print(f'Sites: {sorted(site_hour_priors.keys())}')
print(f'Suppress factor: {SUPPRESS_FACTOR}  (1.0 = disabled)')

Sites: ['S03', 'S08', 'S09', 'S13', 'S15', 'S18', 'S19', 'S22', 'S23']
Suppress factor: 1.0  (1.0 = disabled)


## 6. Load Embeddings

In [7]:
print(f'Loading {EMB_CACHE} ...')
cache       = np.load(EMB_CACHE, allow_pickle=True)
all_emb     = cache['embeddings']
all_logits  = cache['logits']
all_labels  = cache['labels']
all_groups  = cache['groups']
all_weights = cache['weights'] if 'weights' in cache.files else np.ones(len(cache['embeddings']), dtype=np.float32)
print(f'Embeddings : {all_emb.shape}')
print(f'Labels     : {all_labels.shape}')
print(f'Weights    : {all_weights.shape}  mean={all_weights.mean():.3f}')

Loading /kaggle/input/datasets/eduarddehelean/birdclef2026-embeddings/embeddings.npz ...
Embeddings : (147587, 1536)
Labels     : (147587, 234)
Weights    : (147587,)  mean=1.000


## 7. Train MLP Probes

In [8]:
from torch.utils.data import DataLoader, TensorDataset
from collections import defaultdict as _dd
import time

# --- Focal features ---
X_raw    = np.concatenate([all_emb, all_logits], axis=1).astype(np.float32)
print(f'Focal features: {X_raw.shape}')
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
del X_raw; gc.collect()
joblib.dump(scaler, PROBE_DIR / 'scaler.pkl')

pca = PCA(n_components=256, random_state=SEED)
X_focal = pca.fit_transform(X_scaled).astype(np.float32)
del X_scaled; gc.collect()
joblib.dump(pca, PROBE_DIR / 'pca.pkl')
print(f'After PCA: {X_focal.shape}  variance={pca.explained_variance_ratio_.sum():.3f}')
y_focal = (all_labels > 0).astype(np.float32)

# --- Soundscape augmentation (ALL labeled soundscapes) ---
def _pe(t):
    h, m, s = t.split(':'); return int(h)*3600 + int(m)*60 + int(s)

clip_label_map = _dd(lambda: np.zeros(NUM_CLASSES, dtype=np.float32))
for _, row in soundscape_labels.iterrows():
    idx = _pe(row['end']) // 5 - 1
    key = (row['filename'], idx)
    for lbl in str(row['primary_label']).split(';'):
        lbl = lbl.strip()
        if lbl in label2idx:
            clip_label_map[key][label2idx[lbl]] = 1.0

X_snd_list, y_snd_list = [], []
all_labeled = sorted(soundscape_labels['filename'].unique())
snd_paths = [TRAIN_SOUNDSCAPES_DIR / fn for fn in all_labeled
             if (TRAIN_SOUNDSCAPES_DIR / fn).exists()]
for path in tqdm(snd_paths, desc='Soundscape aug'):
    try:
        emb, logits, _, _ = load_soundscape_embeddings(path)
    except Exception as e:
        print(f'  SKIP {path.name}: {e}'); continue
    X_clip = pca.transform(scaler.transform(
        np.concatenate([emb, logits], axis=1).astype(np.float32)))
    for ci in range(len(X_clip)):
        lv = clip_label_map.get((path.name, ci), np.zeros(NUM_CLASSES, dtype=np.float32))
        X_snd_list.append(X_clip[ci])
        y_snd_list.append(lv.copy())

if X_snd_list:
    X_snd = np.array(X_snd_list, dtype=np.float32)
    y_snd = np.array(y_snd_list, dtype=np.float32)
    print(f'Soundscape clips: {len(X_snd)}  positives: {y_snd.sum():.0f}')
    SND_REPEAT = 15
    X_train = np.vstack([X_focal] + [X_snd] * SND_REPEAT)
    y_train = np.vstack([y_focal] + [y_snd] * SND_REPEAT)
else:
    print('No soundscape clips — focal only')
    X_train, y_train = X_focal, y_focal

print(f'Total training clips: {len(X_train)}')

# --- MLP probe ---
pos_counts = y_train.sum(0).clip(min=1)
neg_counts = (len(y_train) - y_train.sum(0)).clip(min=1)
pos_weight = torch.from_numpy(neg_counts / pos_counts).to(TORCH_DEVICE)

loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train),
                  torch.ones(len(y_train), dtype=torch.float32)),
    batch_size=2048, shuffle=True, num_workers=0, pin_memory=True)

linear_probes = nn.Sequential(
    nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(128, NUM_CLASSES)
).to(TORCH_DEVICE)
optimizer = torch.optim.AdamW(linear_probes.parameters(), lr=1e-2, weight_decay=0.1)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

linear_probes.train()
t0 = time.time()
for epoch in range(200):
    total_loss = 0.0
    for xb, yb, _ in loader:
        xb, yb = xb.to(TORCH_DEVICE, non_blocking=True), yb.to(TORCH_DEVICE, non_blocking=True)
        optimizer.zero_grad()
        loss = criterion(linear_probes(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 50 == 0:
        print(f'  epoch {epoch+1:3d}/200  loss={total_loss/len(loader):.4f}  ({time.time()-t0:.0f}s)')

linear_probes.eval()
torch.save(linear_probes.state_dict(), PROBE_DIR / 'linear_probes.pt')
print(f'Done in {time.time()-t0:.1f}s')

Focal features: (147587, 1739)
After PCA: (147587, 256)  variance=0.736


Soundscape aug:   0%|          | 0/66 [00:00<?, ?it/s]

Soundscape clips: 792  positives: 3122
Total training clips: 159467
  epoch  50/200  loss=0.1979  (130s)
  epoch 100/200  loss=0.2946  (261s)
  epoch 150/200  loss=0.2105  (392s)
  epoch 200/200  loss=0.1781  (523s)
Done in 523.4s


## 8. Inference on Test Soundscapes

In [9]:
scaler = joblib.load(PROBE_DIR / 'scaler.pkl')
pca    = joblib.load(PROBE_DIR / 'pca.pkl')
linear_probes = nn.Sequential(
    nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(128, NUM_CLASSES)
).to(TORCH_DEVICE)
linear_probes.load_state_dict(torch.load(PROBE_DIR / 'linear_probes.pt', map_location=TORCH_DEVICE))
linear_probes.eval()
print('Loaded scaler, PCA, linear_probes.pt')

test_files = sorted(TEST_SOUNDSCAPES_DIR.glob('*.ogg'))
print(f'Test soundscapes: {len(test_files)}')

rows = []
for path in tqdm(test_files, desc='Inference'):
    try:
        emb, logits, proxy_logits, _ = load_soundscape_embeddings(path)
    except Exception as e:
        print(f'  skip {path.name}: {e}'); continue

    X_pca = pca.transform(scaler.transform(np.concatenate([emb, logits], axis=1).astype(np.float32)))
    with torch.no_grad():
        scores_all = torch.sigmoid(linear_probes(torch.from_numpy(X_pca).to(TORCH_DEVICE))).cpu().numpy()

    perch_probs = expit(logits)
    proxy_probs = expit(proxy_logits) if proxy_logits.shape[1] > 0 else None
    site, hour_start = parse_site_hour(path.name)

    for clip_idx in range(len(emb)):
        end_sec = (clip_idx + 1) * 5
        hour    = (hour_start + clip_idx * 5 // 3600) % 24
        scores  = scores_all[clip_idx].copy()

        for i, lbl in enumerate(MATCHED_LABELS):
            ci = label2idx[lbl]
            scores[ci] = (1 - PERCH_BLEND) * scores[ci] + PERCH_BLEND * perch_probs[clip_idx, i]
        if proxy_probs is not None:
            for i, lbl in enumerate(GENUS_PROXY_LABELS):
                ci = label2idx[lbl]
                scores[ci] = (1 - PERCH_BLEND) * scores[ci] + PERCH_BLEND * proxy_probs[clip_idx, i]

        scores = apply_prior_suppression(scores[np.newaxis, :], site, hour, site_hour_priors).squeeze()

        row = {'row_id': f'{path.stem}_{end_sec}'}
        row.update(dict(zip(TARGET_CLASSES, scores.tolist())))
        rows.append(row)

sub = pd.DataFrame(rows).reindex(columns=sample_sub.columns, fill_value=0.0)
sub.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print(f'submission.csv — {len(sub)} rows x {len(sub.columns)} cols')
sub.head()

Loaded scaler, PCA, linear_probes.pt
Test soundscapes: 0


Inference: 0it [00:00, ?it/s]

submission.csv — 0 rows x 235 cols


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
